In [2]:
import pandas as pd

# Load the corrupted CSV
df = pd.read_csv('../data/big5_matches.csv', low_memory=False)

print(f"Shape: {df.shape}")
print(f"Total columns: {len(df.columns)}")

# Check for column misalignment - columns that shouldn't exist next to each other
print("\nFirst 3 rows of first 10 columns:")
print(df.iloc[:3, :10].to_string())

# Check if any rows have data in wrong columns
# Look for non-null values in columns that should be empty for most rows
null_counts = df.isnull().sum()
rare_cols = null_counts[null_counts > len(df) * 0.9]  # >90% null
print(f"\nColumns with >90% null values: {len(rare_cols)}")
if len(rare_cols) > 0:
    print(rare_cols.head(20))

# Check if column names look like data values
print("\nSample of column names:")
for i, col in enumerate(df.columns[:50]):
    print(f"  {i}: '{col}'")

# Check for rows where home_team contains a URL or league name (sign of misalignment)
if 'home_team' in df.columns:
    bad_rows = df[df['home_team'].str.contains('http|Premier|Ligue|Bundesliga|Serie|Liga', na=False, regex=True)]
    if len(bad_rows) > 0:
        print(f"\n⚠ Found {len(bad_rows)} rows where home_team looks wrong:")
        print(bad_rows[['home_team', 'away_team', 'match_id']].head(10).to_string())

# Check if columns are repeated
col_counts = pd.Series(df.columns).value_counts()
dupes = col_counts[col_counts > 1]
if len(dupes) > 0:
    print(f"\n⚠ Duplicate columns found:")
    print(dupes)

Shape: (21629, 88)
Total columns: 88

First 3 rows of first 10 columns:
                   league     season                                                                                                   url  match_id    home_team       away_team  Home_possession  Away_possession  Home_passing_accuracy  Away_passing_accuracy
0  England Premier League  2014-2015  https://fbref.com/en/matches/f9755fb7/Southampton-West-Bromwich-Albion-August-23-2014-Premier-League  f9755fb7  Southampton       West Brom               58               42                   81.0                   75.0
1  England Premier League  2014-2015               https://fbref.com/en/matches/482ab595/Liverpool-Arsenal-December-21-2014-Premier-League  482ab595    Liverpool         Arsenal               64               37                   84.0                   72.0
2  England Premier League  2014-2015        https://fbref.com/en/matches/6bf50814/Aston-Villa-Crystal-Palace-January-1-2015-Premier-League  6bf50814  Asto

In [4]:
import pandas as pd

df = pd.read_csv('../data/big5_matches.csv', low_memory=False)

# Check rows where key columns are null
key_nulls = df[df['home_team'].isnull() | df['away_team'].isnull()]
print(f"Rows with null home_team or away_team: {len(key_nulls)}")

# Check for values leaking into wrong columns
# Look at numeric columns for non-numeric values
numeric_cols = ['Home_possession', 'Away_possession', 'Home_minutes', 'Home_goals']
for col in numeric_cols:
    if col in df.columns:
        # Try to find rows where this numeric column has text
        mask = pd.to_numeric(df[col], errors='coerce').isnull() & df[col].notnull()
        bad = df[mask]
        if len(bad) > 0:
            print(f"\n⚠ Column '{col}' has {len(bad)} non-numeric values:")
            print(bad[[col, 'home_team', 'match_id']].head(10).to_string())

# Check for duplicate match_ids
dupes = df[df.duplicated(subset=['match_id'], keep=False)]
if len(dupes) > 0:
    print(f"\n⚠ {len(dupes)} rows with duplicate match_ids:")
    print(dupes[['match_id', 'league', 'season', 'home_team', 'away_team']].head(10).to_string())

# Check the last few rows for completeness
print(f"\nLast 5 rows (all columns):")
print(df.tail(5).to_string())

# Check all columns and their non-null counts
print(f"\nAll {len(df.columns)} columns with non-null counts:")
for i, col in enumerate(df.columns):
    nn = df[col].notna().sum()
    print(f"  {i:3d}: {col} ({nn}/{len(df)} non-null)")

Rows with null home_team or away_team: 0

⚠ Column 'Home_minutes' has 1 non-numeric values:
      Home_minutes   home_team  match_id
11838        1,320  Düsseldorf  723ffa45

⚠ Column 'Home_goals' has 4 non-numeric values:
      Home_goals      home_team  match_id
7312       1,320  Saint-Étienne  78ed8c0d
8087       1,320           Metz  7f01f697
8425       1,320          Reims  5e063c64
12416      1,213   Paderborn 07  36b944ce

Last 5 rows (all columns):
              league     season                                                                                  url  match_id   home_team        away_team  Home_possession  Away_possession  Home_passing_accuracy  Away_passing_accuracy  Home_shots_on_target  Away_shots_on_target  Home_saves  Away_saves  Home_cards  Away_cards  Home_fouls  Away_fouls  Home_corners  Away_corners  Home_crosses  Away_crosses  Home_touches  Away_touches  Home_tackles  Away_tackles Home_interceptions  Away_interceptions Home_aerials_won  Away_aerials_won  

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/big5_matches.csv', low_memory=False)

# Check for obviously wrong values in each column
print("Looking for misaligned values...\n")

# Home_goals should be 0-15 or so, not 990
if 'Home_goals' in df.columns:
    bad_goals = df[pd.to_numeric(df['Home_goals'], errors='coerce') > 20]
    print(f"Home_goals > 20: {len(bad_goals)} rows")
    if len(bad_goals) > 0:
        print(bad_goals[['Home_goals', 'home_team', 'match_id']].head(10))

# Home_cards should be 0-10, not 990
if 'Home_cards' in df.columns:
    bad_cards = df[pd.to_numeric(df['Home_cards'], errors='coerce') > 15]
    print(f"\nHome_cards > 15: {len(bad_cards)} rows")
    if len(bad_cards) > 0:
        print(bad_cards[['Home_cards', 'home_team', 'match_id']].head(10))

# Home_cards_yellow should be 0-10
if 'Home_cards_yellow' in df.columns:
    bad_yellow = df[pd.to_numeric(df['Home_cards_yellow'], errors='coerce') > 15]
    print(f"\nHome_cards_yellow > 15: {len(bad_yellow)} rows")
    if len(bad_yellow) > 0:
        print(bad_yellow[['Home_cards_yellow', 'home_team', 'match_id']].head(10))

# Minutes should be ~990 for team total
if 'Home_minutes' in df.columns:
    # Good values are close to 990 (team total minutes)
    mins = pd.to_numeric(df['Home_minutes'], errors='coerce')
    # Values that are NOT ~990 and NOT NaN
    bad_mins = df[(mins < 900) | (mins > 1000)]
    bad_mins = bad_mins[bad_mins['Home_minutes'].notna()]
    print(f"\nHome_minutes not ~990: {len(bad_mins)} rows")
    if len(bad_mins) > 0:
        print(bad_mins[['Home_minutes', 'home_team', 'match_id']].head(10))

# Check if match_id appears to be corrupted (shifted)
# A valid match_id is 8 hex characters
import re
if 'match_id' in df.columns:
    bad_ids = df[~df['match_id'].astype(str).str.match(r'^[a-f0-9]{8}$', na=False)]
    print(f"\nInvalid match_ids: {len(bad_ids)}")
    if len(bad_ids) > 0:
        print(bad_ids[['match_id', 'home_team']].head(10))

# Check if any numeric columns contain team names or URLs
text_cols = ['Home_possession', 'Home_goals', 'Home_shots', 'Home_minutes']
for col in text_cols:
    if col in df.columns:
        mask = pd.to_numeric(df[col], errors='coerce').isnull() & df[col].notna()
        bad = df[mask]
        if len(bad) > 0:
            print(f"\n{col} has {len(bad)} non-numeric values:")
            print(bad[[col, 'match_id']].head(10).to_string())

Looking for misaligned values...

Home_goals > 20: 10338 rows
     Home_goals         home_team  match_id
2285        990            Fulham  430d879c
2286        990           Everton  66f16cd5
2299        990      Leeds United  c4ed64b9
2301        990         Tottenham  81a7befa
2315        990         Liverpool  81e8aaf4
2316        990         Newcastle  27601c16
2317        990    Crystal Palace  54b07679
2322        990  Sheffield United  1ef0a8f2
2325        990          Brighton  1e2ba709
2332        990    Manchester Utd  d8018048

Home_cards > 15: 2197 rows
      Home_cards       home_team  match_id
2325        17.0        Brighton  1e2ba709
2376        16.0       Liverpool  20bdebdc
2389        23.0  Crystal Palace  7b21d5b9
2403        21.0    Leeds United  8422804d
2511        17.0       West Brom  378a8a95
2633        20.0     Southampton  5e4ec091
2672        19.0         Watford  70fda3e9
2673        17.0        Brighton  8cabd787
2690        16.0    Leeds United  1ca89

In [6]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('../data/big5_matches.csv', low_memory=False)
print(f"Loaded: {df.shape}")

# ============================================================
# STEP 1: Fix comma-formatted numbers
# ============================================================
for col in df.columns:
    if col not in ['league', 'season', 'url', 'match_id', 'home_team', 'away_team']:
        df[col] = df[col].astype(str).str.replace(',', '', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# ============================================================
# STEP 2: Identify correctly-aligned rows (early rows before corruption)
# ============================================================
# A row is "clean" if Home_minutes is ~990 (team total minutes) or NaN (older match)
# A row is "corrupted" if Home_minutes is way off (like 0, 2, 864, 888)

df['_row_idx'] = range(len(df))

# Clean rows: Home_minutes is between 900-1000 OR NaN
clean_mask = (
    (df['Home_minutes'].isna()) | 
    ((df['Home_minutes'] >= 900) & (df['Home_minutes'] <= 1000))
)
clean_rows = df[clean_mask]
corrupt_rows = df[~clean_mask]

print(f"Clean rows: {len(clean_rows)}")
print(f"Corrupt rows: {len(corrupt_rows)}")

# ============================================================
# STEP 3: Find the "break point" where corruption starts
# ============================================================
if len(corrupt_rows) > 0:
    first_corrupt = corrupt_rows['_row_idx'].min()
    print(f"First corrupt row index: {first_corrupt}")
    print(f"Corruption affects rows {first_corrupt} to {len(df)-1}")
    
    # Check what the data looks like just before corruption
    print(f"\nLast clean row ({first_corrupt-1}):")
    last_clean = df.iloc[first_corrupt - 1]
    for col in ['Home_goals', 'Home_assists', 'Home_shots', 'Home_minutes', 'Home_cards_yellow', 'Home_fouled']:
        print(f"  {col}: {last_clean[col]}")
    
    print(f"\nFirst corrupt row ({first_corrupt}):")
    first_bad = df.iloc[first_corrupt]
    for col in ['Home_goals', 'Home_assists', 'Home_shots', 'Home_minutes', 'Home_cards_yellow', 'Home_fouled']:
        print(f"  {col}: {first_bad[col]}")

# ============================================================
# STEP 4: For corrupt rows, try to realign using known valid column positions
# ============================================================
# The correct column order for the player stats section should be:
# minutes, goals, assists, pens_made, pens_att, shots, cards_yellow, cards_red, 
# fouls, fouled, offsides, crosses, tackles_won, interceptions, own_goals, pens_won, pens_conceded

player_stat_cols = [
    'Home_minutes', 'Home_goals', 'Home_assists', 'Home_pens_made', 'Home_pens_att',
    'Home_shots', 'Home_cards_yellow', 'Home_cards_red', 'Home_fouls', 'Home_fouled',
    'Home_offsides', 'Home_crosses', 'Home_tackles_won', 'Home_interceptions', 
    'Home_own_goals', 'Home_pens_won', 'Home_pens_conceded'
]

# For each corrupt row, try to find 990 (minutes) and realign from there
fixed_count = 0
for idx in corrupt_rows.index:
    row = df.loc[idx]
    
    # Find which column has 990 (the minutes column was misaligned)
    found_minutes_col = None
    for col in player_stat_cols:
        val = row[col]
        if pd.notna(val) and 985 <= val <= 995:
            found_minutes_col = col
            break
    
    if found_minutes_col and found_minutes_col != 'Home_minutes':
        # We found the minutes value in the wrong column
        # Calculate the shift amount
        correct_order = player_stat_cols
        actual_start = correct_order.index('Home_minutes')
        found_start = correct_order.index(found_minutes_col)
        shift = found_start - actual_start
        
        if shift > 0:
            # Shift values back to correct columns
            for i in range(len(correct_order) - shift):
                src_col = correct_order[i + shift]
                dst_col = correct_order[i]
                if src_col in df.columns and dst_col in df.columns:
                    df.loc[idx, dst_col] = row[src_col]
            
            # The last 'shift' columns should be NaN (were shifted off the end)
            for i in range(len(correct_order) - shift, len(correct_order)):
                col = correct_order[i]
                if col in df.columns:
                    df.loc[idx, col] = np.nan
            
            fixed_count += 1

print(f"\nFixed {fixed_count} corrupt rows via realignment")

# ============================================================
# STEP 5: Drop helper column and save
# ============================================================
df = df.drop(columns=['_row_idx'])

# Drop sparse/unreliable columns
cols_to_drop = ['Home_shirtnumber', 'Home_nationality', 'Home_position', 'Home_age',
                'Away_shirtnumber', 'Away_nationality', 'Away_position', 'Away_age']
for col in cols_to_drop:
    if col in df.columns:
        df = df.drop(columns=[col])

# Drop duplicates
df = df.drop_duplicates(subset=['match_id'], keep='first')

output_path = '../data/big5_matches_fixed.csv'
df.to_csv(output_path, index=False)
print(f"\nSaved fixed CSV: {output_path}")
print(f"Final shape: {df.shape}")

# Quick sanity check
print(f"\nSanity check - max values:")
for col in ['Home_goals', 'Home_cards_yellow', 'Home_minutes', 'Home_shots']:
    if col in df.columns:
        print(f"  {col}: max={df[col].max()}, min={df[col].min()}")

Loaded: (21629, 88)
Clean rows: 20476
Corrupt rows: 1153
First corrupt row index: 583
Corruption affects rows 583 to 21628

Last clean row (582):
  Home_goals: 2.0
  Home_assists: 2.0
  Home_shots: 18.0
  Home_minutes: 990.0
  Home_cards_yellow: 1.0
  Home_fouled: 12.0

First corrupt row (583):
  Home_goals: 0.0
  Home_assists: 0.0
  Home_shots: 13.0
  Home_minutes: 864.0
  Home_cards_yellow: 1.0
  Home_fouled: 9.0

Fixed 36 corrupt rows via realignment

Saved fixed CSV: ../data/big5_matches_fixed.csv
Final shape: (21629, 80)

Sanity check - max values:
  Home_goals: max=1320.0, min=0.0
  Home_cards_yellow: max=37.0, min=0.0
  Home_minutes: max=1320.0, min=0.0
  Home_shots: max=47.0, min=0.0


In [9]:
import json
import pandas as pd
import numpy as np

# Load checkpoint to get completed URLs
with open('../scraperfc_data/checkpoints/match_scraping_checkpoint.json', 'r') as f:
    checkpoint = json.load(f)

# Load the fixed CSV to find corrupt match_ids
df = pd.read_csv('../data/big5_matches_fixed.csv', low_memory=False)

# Identify corrupt match_ids
corrupt_mask = (
    df['Home_minutes'].notna() & 
    ((df['Home_minutes'] < 900) | (df['Home_minutes'] > 1000))
)
corrupt_match_ids = set(df.loc[corrupt_mask, 'match_id'].tolist())
print(f"Corrupt match_ids: {len(corrupt_match_ids)}")

# Remove these from checkpoint so they get re-scraped
original_completed = len(checkpoint['completed_urls'])
checkpoint['completed_urls'] = [
    url for url in checkpoint['completed_urls'] 
    if url.split('/')[-2] not in corrupt_match_ids
]
new_completed = len(checkpoint['completed_urls'])
print(f"Removed {original_completed - new_completed} URLs from checkpoint")

# Save updated checkpoint
with open('../scraperfc_data/checkpoints/match_scraping_checkpoint.json', 'w') as f:
    json.dump(checkpoint, f, indent=2)

print("Checkpoint updated. Re-run the scraper to fix corrupt matches.")

# Also remove corrupt rows from the CSV so they get replaced
clean_df = df[~df['match_id'].isin(corrupt_match_ids)]
print(f"Clean rows remaining: {len(clean_df)}")

# Save cleaned CSV (without corrupt rows)
clean_df.to_csv('../data/big5_matches_fixed.csv', index=False)
print("Saved cleaned CSV (corrupt rows removed, will be re-scraped)")

Corrupt match_ids: 1117
Removed 1117 URLs from checkpoint
Checkpoint updated. Re-run the scraper to fix corrupt matches.
Clean rows remaining: 20512
Saved cleaned CSV (corrupt rows removed, will be re-scraped)
